# NeuralHub chat playground: lyraaaa/qwen35-mixed
Select **Runtime > Change runtime type > T4 GPU**, then run the cells in order. The last cell opens a chat with steering applied to every reply; conversation history is preserved. Thinking is disabled by default for templates that support it. Settings are prefilled and editable. T4 automatically uses FP16; GPUs with native BF16 use BF16. The full unquantized model must fit in memory; larger or MoE models may not fit T4. Restricted checkpoints need Hugging Face login.

**Experimental defaults, not verified reproduction:** the current model branch is resolved to an exact commit once per session and printed. The default operation is raw activation addition. Saved doses are shown below, but source methods may use different normalization units (including the neuron-explorer residual-norm unit); this SDK uses absolute activation operands. Confirm the original method before interpreting effects.

In [ ]:
%pip install 'neuralhub-sdk[transformers] @ https://mywalnuts.com/research/neuralhub/api/sdk/source.zip' 'gradio>=6,<7'

In [ ]:
from neuralhub import Neuron, Model

HUB_URL = 'https://mywalnuts.com/research/neuralhub'
REPOSITORY = 'lyraaaa/qwen35-mixed'
REVISION = 1
MODEL_REPOSITORY = 'Qwen/Qwen3.5-4B'

# Ready-to-run exploratory settings. Edit these to try another experiment.
MODEL_REVISION = "main"  # Resolved once to an exact HF commit before loading.
OPERATION = "add"        # Experimental raw addition at mlp.down_proj.input.
STRENGTH = 'per_coordinate'  # Saved dose, or "per_coordinate" for the list below.
COORDINATE_STRENGTHS = [0.6, 0.6, -0.5, 0.4, -0.3]  # Coordinate order matches the pinned recipe.
MISSING_DOSE_COORDINATES = []  # Missing doses default to experimental +1.0.
ALLOW_APPROXIMATE = False  # Uses the listed approximate coordinate when True.
DEVICE = "auto"  # CUDA GPU; set "cpu" explicitly for slow local CPU execution.
DTYPE = "auto"   # Native BF16 on supported GPUs, FP16 on T4, FP32 on CPU.
PROMPT = "Tell me a short story."
MAX_NEW_TOKENS = 512
ENABLE_THINKING = False
MAX_INPUT_TOKENS = 4096  # Start a new chat when this limit is reached.


def runtime_settings():
    import torch
    device = DEVICE
    if device == "auto":
        if not torch.cuda.is_available():
            raise RuntimeError("Choose a GPU in Colab: Runtime > Change runtime type > T4 GPU, then rerun. For local CPU execution set DEVICE = 'cpu'.")
        device = "cuda"
    if DTYPE == "auto":
        if str(device).startswith("cuda"):
            native_bf16 = torch.cuda.get_device_capability(device)[0] >= 8 and torch.cuda.is_bf16_supported()
            dtype = torch.bfloat16 if native_bf16 else torch.float16
        else:
            dtype = torch.float32
    else:
        if DTYPE not in ("float16", "bfloat16", "float32"):
            raise ValueError("DTYPE must be auto, float16, bfloat16 or float32")
        dtype = getattr(torch, DTYPE)
    return device, dtype


def load_experiment():
    import math
    import re
    from huggingface_hub import model_info
    global MODEL_REVISION
    if not isinstance(MODEL_REVISION, str) or not MODEL_REVISION:
        raise ValueError("MODEL_REVISION must be a model branch, tag or exact commit")
    if OPERATION not in ("add", "scale", "set"):
        raise ValueError("OPERATION must be add, scale or set")
    doses = COORDINATE_STRENGTHS if STRENGTH == "per_coordinate" else [STRENGTH]
    if not doses or any(type(d) not in (int, float) or not math.isfinite(d) or abs(d) > 100 for d in doses):
        raise ValueError("Strengths must be finite numbers between -100 and 100")
    device, dtype = runtime_settings()
    neuron = Neuron.from_hub(REPOSITORY, base_url=HUB_URL, revision=REVISION)
    if neuron.document["model"].get("huggingface") != MODEL_REPOSITORY:
        raise ValueError("The recipe model differs from the exported notebook")
    if not ALLOW_APPROXIMATE and any(p.get("coordinatePrecision") == "approximate" for p in neuron.document["positions"]):
        raise ValueError("This recipe uses approximate coordinates; set ALLOW_APPROXIMATE = True to use them as listed.")
    if STRENGTH == "per_coordinate":
        if len(COORDINATE_STRENGTHS) != len(neuron.document["positions"]):
            raise ValueError("Provide one dose for every recipe coordinate")
        neuron = Neuron({**neuron.document, "positions": [
            {**p, "strength": dose} for p, dose in zip(neuron.document["positions"], COORDINATE_STRENGTHS)
        ]})
    if neuron.document.get("format") == "neuralhub/evidence-v1":
        # Explicit exploratory raw operands, not a conversion of source units.
        neuron = neuron.prepare(
            positions=neuron.document["positions"] if STRENGTH == "per_coordinate" else None,
            strength=None if STRENGTH == "per_coordinate" else STRENGTH)
    if not re.fullmatch(r"[0-9a-fA-F]{40}", MODEL_REVISION):
        resolved = model_info(MODEL_REPOSITORY, revision=MODEL_REVISION).sha
        if not isinstance(resolved, str) or not re.fullmatch(r"[0-9a-fA-F]{40}", resolved):
            raise ValueError("Hugging Face did not return an exact model commit")
        MODEL_REVISION = resolved  # Reused for subsequent runs in this session.
    print(f"Model: {MODEL_REPOSITORY} @ {MODEL_REVISION}\nDevice: {device}; dtype: {dtype}")
    print(f"Recipe: {REPOSITORY} r{REVISION}; operation: {OPERATION}; strengths: {doses}")
    print("Experimental settings: the current model checkpoint and raw activation doses do not establish reproduction of the source method or effect.")
    if MISSING_DOSE_COORDINATES:
        print(f"Missing source doses at coordinate indices {MISSING_DOSE_COORDINATES} were prefilled with experimental +1.0; review before interpreting results.")
    if ALLOW_APPROXIMATE:
        print("Approximate coordinates are enabled and will be used as listed.")
    model = Model.from_neuron(neuron, model_revision=MODEL_REVISION, device=device, dtype=dtype, attn_implementation="sdpa")
    return neuron, model


def compare():
    neuron, model = load_experiment()
    options = dict(do_sample=False, max_new_tokens=MAX_NEW_TOKENS, enable_thinking=ENABLE_THINKING)
    baseline = model.generate(PROMPT, **options)
    with model.steer(neuron, operation=OPERATION,
                     strength=None if STRENGTH == "per_coordinate" else STRENGTH,
                     allow_approximate=ALLOW_APPROXIMATE):
        steered = model.generate(PROMPT, **options)
    print("BASELINE\n" + baseline + "\n\nSTEERED\n" + steered)
    return {"baseline": baseline, "steered": steered, "model_revision": MODEL_REVISION}



In [ ]:
import gradio as gr
from threading import Lock

# Run this cell once. The same model is reused for every chat turn.
neuron, model = load_experiment()
generation_lock = Lock()


def chat(message, history, strength_multiplier, temperature, max_tokens):
    messages = []
    for turn in history:
        content = turn["content"]
        if isinstance(content, list):
            content = "".join(block.get("text", "") for block in content
                              if isinstance(block, dict) and block.get("type") == "text")
        if turn["role"] in ("user", "assistant") and isinstance(content, str):
            messages.append({"role": turn["role"], "content": content})
    messages.append({"role": "user", "content": message})
    doses = COORDINATE_STRENGTHS if STRENGTH == "per_coordinate" else [STRENGTH] * len(neuron.document["positions"])
    scaled = [dose * strength_multiplier for dose in doses]
    if any(abs(dose) > 100 for dose in scaled):
        raise gr.Error("Resulting strength exceeds 100. Reduce the strength multiplier.")
    active = Neuron({**neuron.document, "positions": [
        {**position, "strength": dose}
        for position, dose in zip(neuron.document["positions"], scaled)
    ]})
    options = dict(max_new_tokens=int(max_tokens), enable_thinking=ENABLE_THINKING,
                   max_input_tokens=MAX_INPUT_TOKENS, do_sample=temperature > 0)
    if temperature > 0:
        options.update(temperature=temperature, top_p=0.9)
    try:
        with generation_lock, model.steer(active, operation=OPERATION,
                                          allow_approximate=ALLOW_APPROXIMATE):
            return model.generate(messages, **options)
    except ValueError as error:
        raise gr.Error(str(error)) from error


with gr.Blocks() as demo:
    gr.Markdown("# NeuralHub · Steered chat")
    gr.Markdown(f"Recipe: `{REPOSITORY}` · r{REVISION} · `{OPERATION}` · saved strength: `{STRENGTH}`. "
                "Every reply uses steering. Strength changes apply to future replies; start a new chat for a fresh experiment.")
    gr.Markdown("Experimental raw activation doses; the source effect has not been verified. "
                "The model runs on your Colab runtime. This temporary Gradio link can be used by anyone who has the link while the runtime is active.")
    with gr.Row():
        strength_control = gr.Slider(-2, 2, value=1, step=0.1, label="Strength multiplier (1 = recipe default)")
        temperature_control = gr.Slider(0, 1.5, value=0.7, step=0.05, label="Temperature (0 = deterministic)")
        length_control = gr.Slider(64, 1024, value=MAX_NEW_TOKENS, step=64, label="Max new tokens")
    gr.ChatInterface(fn=chat,
                     additional_inputs=[strength_control, temperature_control, length_control],
                     concurrency_limit=1)

# Colab requires a temporary tunnel to display the interactive chat.
demo.queue(default_concurrency_limit=1).launch(share=True, inline=True, show_error=True)
